# NOMINA: Generative AI for Pharmaceutical Naming

This notebook runs the complete **NOMINA Generator + Verifier architecture**. It allows you to select the generation mode, apply hard regulatory verifications, and run the automated refinement loop.

In [ ]:
!pip install -q openai pandas pydantic numpy

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get('llmkey')
    print("API key 'llmkey' loaded from Colab secrets.")
except ImportError:
    print("Not running in Colab. Ensure you have set your OPENROUTER_API_KEY environment variable if you plan to use llm_baseline.")

## 1. Setup the Codebase
Clone the repository to get access to the data layer, the verifier, and the generator.

In [ ]:
import os
if not os.path.exists("Pharmaceutical-Name-Generation"):
    !git clone https://github.com/vedanshshetty/Pharmaceutical-Name-Generation.git
    
import sys
if "Pharmaceutical-Name-Generation" not in sys.path:
    sys.path.append("Pharmaceutical-Name-Generation")

import data_layer
from verifier import Verifier
from generator import Generator, GeneratorConfig

# Note: Change branches inside the cloned folder if your final code is on a specific branch, 
# e.g., !cd Pharmaceutical-Name-Generation && git checkout Generator

## 2. Initialize the Architecture
Both the Generator and Verifier load their reference datasets from the shared data layer.

In [ ]:
print("Initializing Verifier and Generator...")
verifier = Verifier.from_data_layer(data_layer)

# We also configure the generator to use the OpenRouter API key we loaded
config = GeneratorConfig()
generator = Generator.from_data_layer(data_layer, config=config)

## 3. Run the Generation & Refinement Pipeline
Configure your target drug class, required stems, and the generation strategy below.

In [ ]:
# ==========================================================
# ⚙️ GENERATOR CONFIGURATION
# ==========================================================

# "generic" (requires a stem) or "brand" (no stem allowed)
TARGET_TYPE = "generic"

CLASS_KEYWORD = "beta-blocker"

# Must match the pharmacological class. 
# You can look this up via data_layer.stems_for_class("beta-blocker")
STEM = "-olol" 

# STRATEGIES:
# 1. "llm_baseline"         (Uses OpenRouter LLM to propose candidates)
# 2. "rejection_sampling"   (Local character n-gram model)
# 3. "constrained_decoding" (Local linguistic syllable rules)
# 4. "rl_refined"           (Local n-gram model + RL feedback from Verifier)
STRATEGY = "llm_baseline" 

# How many successful names to produce before stopping
N_ACCEPTED = 5 

# ==========================================================

import os
from generator import export_generation_report

api_key = os.environ.get("OPENROUTER_API_KEY")

print(f"Starting pipeline: target={TARGET_TYPE}, class={CLASS_KEYWORD}, strategy={STRATEGY}")

# Note: The 'generator.generate_and_refine' handles LLM injection natively if we update it, 
# but for safety we can directly patch the environment variable if we are using openai.
if STRATEGY == "llm_baseline":
    if not api_key:
        print("\n⚠️ WARNING: You selected 'llm_baseline' but 'llmkey' is not set in Colab Secrets.")
    
results, stats = generator.generate_and_refine(
    verifier=verifier,
    n_accepted=N_ACCEPTED,
    target_type=TARGET_TYPE,
    target_class=CLASS_KEYWORD,
    target_stem=STEM,
    strategy=STRATEGY
)

print("\n" + "="*40)
print("📊 GENERATION STATS")
print("="*40)
for k, v in stats.items():
    print(f"{k}: {v}")
    
print("\n" + "="*40)
print("✅ ACCEPTED CANDIDATES")
print("="*40)
for r in results:
    if r.accepted:
        print(f"- {r.candidate_name}")
        print(f"  ↳ Rounds: {r.rounds}")
        print(f"  ↳ Lineage: {' -> '.join(r.lineage)}")
        if r.final_response:
            print(f"  ↳ Risk Score: {r.final_response.composite_risk_score}")
        print()
        
accepted_csv, all_csv = export_generation_report(results, path_prefix="/content/nomina_results")
print(f"\nSaved output to {accepted_csv} and {all_csv}")